# BA 777 - Business Analytics Toolbox
## H&M Personalized Fashion Recommendations EDA Practice Exercise Version


## Introduction

[H&M Personalized Fashion Recommendations](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/overview)

Welcome to our H&M Exploratory Data Analysis notebook. Our focus is to explore the intricate patterns and insights within the H&M retail ecosystem, leveraging three key data sources: articles.csv, customers.csv, and transactions_train.csv. These files offer a comprehensive look at product metadata, customer demographics, and transactional history, respectively. Although our analysis is independent of the competition's scope to develop product recommendations, the insights gleaned here could provide a foundational understanding that is crucial for any advanced predictive modeling. Let's embark on this data exploration journey, unraveling the stories told by the data through the powerful lens of SQL.

# Preparation

## Copy the Dataset

In [ ]:
from google.colab import auth
auth.authenticate_user()

To speed things up, we'll copy the dataset directly from the course bucket into your personal bucket. Otherwise, you would need to create a Kaggle API key, download the dataset from Kaggle, and then upload it to your bucket, which takes longer.

Replace the placeholders below with your project ID and bucket name. Your bucket should be in the US, the same location as your BigQuery data. Setting `GOOGLE_CLOUD_PROJECT` here means you don't need `--project` on every SQL cell.

In [ ]:
import os

your_project = 'YOUR_PROJECT_ID'
your_bucket = 'YOUR_BUCKET'

os.environ['GOOGLE_CLOUD_PROJECT'] = your_project

In [ ]:
!gcloud storage cp gs://ba-777-public/data/h_and_m/*.csv gs://$your_bucket/data/h_and_m/

Verify the files were copied successfully:

In [ ]:
!gcloud storage ls --long --readable-sizes gs://$your_bucket/data/h_and_m/

Create a new dataset (schema) called `h_and_m`:

In [ ]:
%%bigquery
CREATE SCHEMA IF NOT EXISTS h_and_m

Create 3 tables from the 3 csv files: `articles.csv`, `customers.csv`, and `transactions_train.csv`.

Make sure to replace **YOUR_BUCKET** with your bucket name:

In [ ]:
%%bigquery
LOAD DATA OVERWRITE h_and_m.articles
FROM FILES (
  format = 'CSV',
  uris = ['gs://YOUR_BUCKET/data/h_and_m/articles.csv']
);

In [ ]:
%%bigquery
LOAD DATA OVERWRITE h_and_m.customers
FROM FILES (
  format = 'CSV',
  uris = ['gs://YOUR_BUCKET/data/h_and_m/customers.csv']
);

We'll save `transactions_train.csv` as `transactions`:

In [ ]:
%%bigquery
LOAD DATA OVERWRITE h_and_m.transactions
FROM FILES (
  format = 'CSV',
  uris = ['gs://YOUR_BUCKET/data/h_and_m/transactions_train.csv']
);

The `transactions` table has over 31 million rows (almost 3 GB), so we will work with a sample during our initial exploration to save cost and improve performance. We keep about 1% of customers, with all of their purchases, so each sampled customer's history stays complete:

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE h_and_m.transactions_sample
AS
SELECT * FROM h_and_m.transactions
WHERE MOD(ABS(FARM_FINGERPRINT(customer_id)),100) = 1

## Initial Inspection
Let's print out three rows of entries for each of the tables so we can have a rough idea what each table looks like.

### Print out 3 rows of the entries for `articles`

In [ ]:
%%bigquery
# YOUR CODE HERE

,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,...,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name,detail_desc
0,662344003,662344,Lily blouse,258,Blouse,Garment Upper body,-1,Unknown,-1,Unknown,...,Blouse,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1010,Blouses,Long-sleeved blouse in satin with a small stan...
1,592986010,592986,Cool tiebelt dressed,272,Trousers,Garment Lower body,-1,Unknown,-1,Unknown,...,Trouser,A,Ladieswear,1,Ladieswear,11,Womens Tailoring,1009,Trousers,"Wide, high-waisted trousers in woven fabric wi..."
2,665665001,665665,J LORENZO TRS EQ,272,Trousers,Garment Lower body,-1,Unknown,-1,Unknown,...,Equatorial Assortment,A,Ladieswear,1,Ladieswear,15,Womens Everyday Collection,1001,Unknown,Leggings in soft jersey made from a viscose bl...


### Print out 3 rows of the entries for `customers`

In [ ]:
%%bigquery
SELECT * FROM h_and_m.customers LIMIT 3

,customer_id,FN,Active,club_member_status,fashion_news_frequency,age,postal_code
0,000064249685c11552da43ef22a5030f35a147f723d5b0...,NaN,NaN,NaN,NaN,<NA>,2c29ae653a9282cce4151bd87643c907644e09541abc28...
1,004d89470677ce579a313b2259e93b07ea6f944686f9c3...,NaN,NaN,NaN,NaN,<NA>,2c29ae653a9282cce4151bd87643c907644e09541abc28...
2,008681443e68c3e60a686217a704187dd38abffd2c0962...,NaN,NaN,NaN,NaN,<NA>,2c29ae653a9282cce4151bd87643c907644e09541abc28...


### Print out 3 rows of the entries for `transactions_sample`

In [ ]:
%%bigquery
SELECT * FROM h_and_m.transactions_sample LIMIT 3

,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,02e45757a6bc483ef7a9d086c3800bdab53ac1cb9840cf...,537688014,0.050831,1
1,2018-09-20,02e45757a6bc483ef7a9d086c3800bdab53ac1cb9840cf...,637515002,0.033881,1
2,2018-09-20,02e45757a6bc483ef7a9d086c3800bdab53ac1cb9840cf...,537346024,0.042356,1


## Basic Exercises

### WHERE

Write a query to find all transactions from the `transactions_sample` table for the date `2019-01-24` whose `price` is above 0.02. Select the columns `customer_id`, `article_id`, and `price` in your result.

In [ ]:
%%bigquery

# YOUR CODE HERE

,customer_id,article_id,price
0,02354fba443ecfd9914e9f842fe8a61e7e367cf6dd7057...,571706001,0.022017
1,02354fba443ecfd9914e9f842fe8a61e7e367cf6dd7057...,571706001,0.022017
2,023c2ece3f5ab273c693a88300951320d47949cda63f11...,732690002,0.067780
3,023c2ece3f5ab273c693a88300951320d47949cda63f11...,639976002,0.040661
4,023c2ece3f5ab273c693a88300951320d47949cda63f11...,732690002,0.067780
...,...,...,...
263,f7972f791a628d6b292a99af32c3d30726e5a6771fdd44...,689389015,0.021593
264,f7972f791a628d6b292a99af32c3d30726e5a6771fdd44...,682305001,0.050831
265,f8a8c5d26ea8dd1471c2008215cdd55e04ca4650166308...,529008010,0.021424
266,fd438ed451591c6d2d4787ad9150bd2d62e5d0c7acaec9...,650500003,0.059305


### GROUP BY

Write a query to calculate the total sales for each `article_id` on `2019-01-24` in the `transactions_sample` table. Display the `article_id` and the sum of `price` as `total_sales`.

In [ ]:
%%bigquery

# YOUR CODE HERE


,article_id,total_sales
0,571706001,0.044034
1,189616001,0.016932
2,732690002,0.135559
3,639976002,0.040661
4,699996001,0.050831
...,...,...
417,679278002,0.018576
418,635190001,0.015237
419,675864001,0.016932
420,650500003,0.059305


### ORDER BY

Write a query to sort the above query by total_sales in descending order.

In [ ]:
%%bigquery

# YOUR CODE HERE

,article_id,total_sales
0,639357001,0.152525
1,752228002,0.135576
2,674071001,0.135559
3,732690002,0.135559
4,677999001,0.127068
...,...,...
417,706268011,0.003390
418,670279001,0.003373
419,649059001,0.003373
420,355569002,0.003373


### HAVING

Write a query to find customers from the `customers` table who have an `age` greater than 30 and are classified as `ACTIVE` in the `club_member_status`. Display their number of purchases and filter on those who have made more than 2 purchases.

In [ ]:
%%bigquery

# YOUR CODE HERE
# HINT: USE SELECT, FROM, JOIN, WHERE, GROUP BY, HAVING in this order




,customer_id,purchase_count
0,5e274ef08c6e8a0c6e5105746987edb5d866ae76b2faad...,8
1,a08552cc29a4463eec4da1dc04da7c57392d0977658eea...,6
2,e6719f4c73386284779b977363f37a972148ca5e4463f1...,5
3,ff3e37f3f23fc8673de710c19ac714c8d20d3fc7ab1fd7...,6
4,0243be4739f6ebf04c9efb04dbb6161264432118b1d78e...,117
...,...,...
5399,324b56b77fb42ed765b97fe75cd9d9a685c7812212bf32...,4
5400,97fb016faa008ebda1d114f3a498e8d4dc0e3e60305b45...,4
5401,ddbef66fcefb45cdd44d6149d178c429f3359fa6071bdb...,7
5402,bb76da09355b7509853ab5ec62585559d7a7631e30eb33...,6


### WITH

Create a query using a WITH clause to establish a Common Table Expression (CTE) named `CustomerSpending`. This CTE should calculate the total spending for each customer from the `transactions_sample` table. Use the `WHERE` clause to filter for customers with a total spending greater than 0.1.

In [ ]:
%%bigquery

# YOUR CODE HERE
# Hint: WITH xxxx AS() then write your select


,customer_id,total_spending
0,02e45757a6bc483ef7a9d086c3800bdab53ac1cb9840cf...,2.199627
1,0384f1a2abd76923b5aba3c1614c6a3845f3b5dd585ca7...,1.184203
2,0864c58e052b243273d2ab33df294f8e03e9299f84a925...,8.384915
3,0e230d363f8f7418d5cba209bff7347c71d7c9c0cc96e7...,0.219508
4,102b78267df2dbc2dd72f5dbc682d9ec85afab66085989...,2.037695
...,...,...
9740,992298beefbddddc3f7122de1532d1ec552706e848cd14...,0.186356
9741,e30d13adefda79f5d7f7c284e05f5f392bc4b4c2264616...,0.191424
9742,f35459c8b8d02cae837aaa4414cb95136dcd97628d726d...,0.123661
9743,32567072ab7628f01fce2f959a9d74708c1089c221c4b2...,0.210407


### Subqueries

Write a query to list all transactions from the `transactions_sample` table where the price of the article is higher than the average price of all articles sold on the same date `(2019-01-24)`. The query should use a subquery to determine the average price for that date. Select the `article_id`, `price`, and `t_dat` for each transaction that meets this condition.

In [ ]:
%%bigquery

# YOUR CODE HERE
# HINT: in your WHERE statement, make your price > (subquery)


,article_id,price,t_dat
0,732690002,0.067780,2019-01-24
1,639976002,0.040661,2019-01-24
2,732690002,0.067780,2019-01-24
3,699996001,0.050831,2019-01-24
4,664461002,0.030492,2019-01-24
...,...,...,...
209,733404001,0.033881,2019-01-24
210,764132001,0.036000,2019-01-24
211,682305001,0.050831,2019-01-24
212,650500003,0.059305,2019-01-24


### Window Function

EX1: Write a query to find the running total of sales for each day in the `transactions_sample` table. Display the transaction date (`t_dat`), article ID (`article_id`), individual price, and a new column named `running_total` that adds up the day's sales as you go down the rows, restarting at each new date. Within a day, add the rows up in order of `article_id`. Sort the final result by date and then `running_total`, both ascending.

Hint:
Use the `SUM()` function with the `OVER()` clause. You will need `PARTITION BY`, `ORDER BY`, and a `ROWS BETWEEN` frame. Without the frame, rows that tie on `article_id` get the same value.

In [ ]:
%%bigquery

# YOUR CODE HERE


,t_dat,article_id,price,running_total
0,2018-09-20,156231001,0.007610,0.007610
1,2018-09-20,179393018,0.013542,0.021153
2,2018-09-20,179393018,0.013542,0.034695
3,2018-09-20,181448102,0.006763,0.041458
4,2018-09-20,186262006,0.004220,0.045678
...,...,...,...,...
317914,2020-09-22,934536001,0.059305,11.586169
317915,2020-09-22,935541001,0.022864,11.609034
317916,2020-09-22,939492001,0.021678,11.630712
317917,2020-09-22,939503001,0.016254,11.646966


EX2: Write a query to rank each customer's purchases by their price for transactions made on '2019-01-24'. Your result should include the customer's ID, the article they purchased, the price of the article, and a rank number with 1 representing their highest-priced purchase.

Hint: To rank something within groups in SQL, you can use the `RANK()` function in combination with `OVER()` and `PARTITION BY`.

In [ ]:
%%bigquery

#YOUR CODE HERE


,customer_id,article_id,price,price_rank
0,02354fba443ecfd9914e9f842fe8a61e7e367cf6dd7057...,571706001,0.022017,1
1,02354fba443ecfd9914e9f842fe8a61e7e367cf6dd7057...,571706001,0.022017,1
2,02354fba443ecfd9914e9f842fe8a61e7e367cf6dd7057...,189616001,0.016932,3
3,023c2ece3f5ab273c693a88300951320d47949cda63f11...,732690002,0.067780,1
4,023c2ece3f5ab273c693a88300951320d47949cda63f11...,732690002,0.067780,1
...,...,...,...,...
480,f8a8c5d26ea8dd1471c2008215cdd55e04ca4650166308...,511783002,0.005712,3
481,fd438ed451591c6d2d4787ad9150bd2d62e5d0c7acaec9...,650500003,0.059305,1
482,fd438ed451591c6d2d4787ad9150bd2d62e5d0c7acaec9...,675864001,0.016932,2
483,fd438ed451591c6d2d4787ad9150bd2d62e5d0c7acaec9...,635190001,0.015237,3


## Exploring the `articles` table

###  Display some of the images

The product images are too large to copy through the course bucket (105,108 files on Kaggle), so here is a saved sample of 36 of them:

<img src="https://raw.githubusercontent.com/soltaniehha/Business-Analytics-Toolbox/master/docs/images/h-and-m-sample.png" width="576">

### How many unique articles/products/etc are in this table?


In [ ]:
%%bigquery

#YOUR CODE HERE

,n_rows,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,...,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name
0,105542,105542,47224,45875,132,131,19,30,30,50,...,299,250,10,10,5,5,57,56,21,21


### Can you investigate and figure out what the difference between `article_id` and `product_code` is?

In [ ]:
%%bigquery

#YOUR CODE HERE

,product_code,n_articles
0,783707,75
1,684021,70
2,699923,52
3,699755,49
4,685604,46
...,...,...
47219,581872,1
47220,687681,1
47221,652192,1
47222,769806,1


### We're interested in the product_code 783707. Let's do some exploration:

Let's first find out the number of unique entries for rows with product_code 783707.

In [ ]:
%%bigquery

#YOUR CODE HERE

,n_rows,article_id,product_code,prod_name,product_type_no,product_type_name,product_group_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,...,department_no,department_name,index_code,index_name,index_group_no,index_group_name,section_no,section_name,garment_group_no,garment_group_name
0,75,75,1,3,1,1,1,3,3,23,...,1,1,1,1,1,1,1,1,1,1


Then, can you use STRING_AGG and CAST to show the content of the distinct entries?

In [ ]:
%%bigquery
#YOUR CODE HERE

,prod_name,graphical_appearance_no,graphical_appearance_name,colour_group_code,colour_group_name,n_detail_desc
0,"1pk Fun 1,1pk Fun,1pk fun sock","1010001,1010022,1010014","All over pattern,Jacquard,Placement print","6,7,8,9,10,13,19,21,32,33,42,52,61,62,63,71,72...","Light Grey,Grey,Dark Grey,Black,White,Beige,Gr...",1


Let's look at the images of this product. They are not in the course bucket either, so here is a saved picture of all of its articles:

<img src="https://raw.githubusercontent.com/soltaniehha/Business-Analytics-Toolbox/master/docs/images/h-and-m-product-783707.png" width="800">

### What is the breakdown of `index_name`?

In [ ]:
%%bigquery
#YOUR CODE HERE

,index_name,n_articles
0,Ladieswear,26001
1,Divided,15149
2,Menswear,12553
3,Children Sizes 92-140,12007
4,Children Sizes 134-170,9214
5,Baby Sizes 50-98,8875
6,Ladies Accessories,6961
7,Lingeries/Tights,6775
8,"Children Accessories, Swimwear",4615
9,Sport,3392


### What is the breakdown of `index_group_name`?

In [ ]:
%%bigquery
#YOUR CODE HERE

,index_group_name,n_articles,n_index_name,index_name,n_garment_group_name,garment_group_name
0,Ladieswear,39737,3,"Ladieswear,Lingeries/Tights,Ladies Accessories",18,"Blouses,Trousers,Unknown,Outdoor,Dressed,Dress..."
1,Baby/Children,34711,4,"Children Accessories, Swimwear,Children Sizes ...",17,"Accessories,Socks and Tights,Swimwear,Jersey B..."
2,Divided,15149,1,Divided,16,"Dresses Ladies,Blouses,Jersey Fancy,Jersey Bas..."
3,Menswear,12553,1,Menswear,16,"Unknown,Shirts,Knitwear,Jersey Fancy,Under-, N..."
4,Sport,3392,1,Sport,5,"Jersey Fancy,Accessories,Unknown,Outdoor,Trousers"


### What is the breakdown of the combination for `index_group_name` and `garment_group_name`?





In [ ]:
%%bigquery
#YOUR CODE HERE

,index_group_name,garment_group_name,n_articles
0,Baby/Children,Jersey Fancy,9103
1,Ladieswear,Jersey Fancy,6163
2,Ladieswear,Accessories,4999
3,Ladieswear,"Under-, Nightwear",4513
4,Ladieswear,Blouses,4492
...,...,...,...
67,Divided,Special Offers,148
68,Divided,Swimwear,76
69,Baby/Children,Unknown,59
70,Menswear,Special Offers,25


## Exploring the `customers` table

### What is the age demographic of the customers?

In [ ]:
%%bigquery
#YOUR CODE HERE

,age,n_customers
0,10s,71583
1,20s,528358
2,30s,234068
3,40s,204118
4,50s,226242
5,60s,72623
6,70s,17611
7,80s,1415
8,90s,101
9,Unknown,15861


## Exploring the `transactions` table

Find out the date for the first and the latest transactions

In [ ]:
%%bigquery
#YOUR CODE HERE

,first_date,latest_date
0,2018-09-20,2020-09-22


Top 10 customers who purchased most often

In [ ]:
%%bigquery
#YOUR CODE HERE

,customer_id,n_transactions
0,d80ed4ababfa96812e22b911629e6bcbf5093769051ea4...,1066
1,e3717b1d03c73d1cfd0f27ef09c32381aab5ca11c5aed9...,592
2,4e57c43d931e365659f756e5b173b79b9fdc8cbc6a5eab...,587
3,578e704b2c2fc32d93261b4be9d4797301518f653a9c17...,511
4,0d8e6c1ea7890ce90968f017361e439f4ea9091de795e9...,503
5,2d545e697d8cf36558c81eb56c1776cb30f893585ed21a...,479
6,66fc1a29a90f01fe4d9b5e6f00d3e6fc803a4b5ffafbc7...,475
7,392d4ed7fe2b4b6bdf61f4ef1b6f9b614ce7c7d8052c60...,456
8,6b5851409a9ca9c54033fea3c354b7277b369fc0e61775...,437
9,2fd690c02e3f0a1def03a331bfbb133f0d0790af4f013f...,437


Top 10 customers who spent the most

In [ ]:
%%bigquery
#YOUR CODE HERE

,customer_id,revenue
0,d80ed4ababfa96812e22b911629e6bcbf5093769051ea4...,39.821627
1,0d8e6c1ea7890ce90968f017361e439f4ea9091de795e9...,21.904220
2,4e57c43d931e365659f756e5b173b79b9fdc8cbc6a5eab...,18.525424
3,e3717b1d03c73d1cfd0f27ef09c32381aab5ca11c5aed9...,18.267475
4,578e704b2c2fc32d93261b4be9d4797301518f653a9c17...,17.440424
5,9e6b6c5bfb5fa347204dbd93f51d84be5fa631a09226ff...,16.356339
6,419d24b577b25ebf4e60feb39ff0466e6d7752f8ba02d1...,15.557136
7,42fd70f36a393442030b07e9fbc74c9fe8aac1769be4b4...,14.417780
8,2fd690c02e3f0a1def03a331bfbb133f0d0790af4f013f...,14.283678
9,6d1d3142514e6376099fa031d1cf16b8fb7a40060d881e...,13.827407


Top 5 revenue generating `product_group_name`

In [ ]:
%%bigquery
#YOUR CODE HERE

,product_group_name,revenue
0,Garment Upper body,3418.268915
1,Garment Lower body,2348.618763
2,Garment Full body,1266.220797
3,Swimwear,577.979271
4,Underwear,537.535017
